In [1]:
# lets start by check torch version 
import torch
print(torch.__version__)

2.10.0


starting with torch 2.10.0

In [3]:
# Is msp available?
torch.mps.is_available()

True

In [4]:
# we are going to use library reasoning from scratch
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch", 
    "torch",
    "tokenizers"
]
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.2.0
torch version: 2.10.0
tokenizers version: 0.23.2


In [9]:
# lets represent text as tokens -> input text as input ids tokens
# first download qwen3 model
from reasoning_from_scratch.qwen3 import download_qwen3_small
    
download_qwen3_small(kind="base", tokenizer_only=True, out_dir="./qwen3_model")


In [11]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
tokenizer_path = Path("./qwen3_model/") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_path)

In [15]:
prompt = "Hello, how are you?"
input_ids = tokenizer.encode(prompt)
for i in input_ids:
    print(f"{i} -> {tokenizer.decode([i])}")

text = tokenizer.decode(input_ids)
print(f"Decoded text: {text}")  

9707 -> Hello
11 -> ,
1246 ->  how
525 ->  are
498 ->  you
30 -> ?
Decoded text: Hello, how are you?


In [24]:
#device = torch.device("mps")
device = torch.device("cpu")
print("Using Apple Silicon GPU (MPS)")

Using Apple Silicon GPU (MPS)


In [18]:
#download_qwen3_small(kind="base", tokenizer_only=False, out_dir="./qwen3_model")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [25]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B
model_path = Path("./qwen3_model/") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [30]:
prompt = "please explain quantum computing"
input_tokens_id_list = tokenizer.encode(prompt)
input_tensor = torch.tensor(input_tokens_id_list).unsqueeze(0).to(device)
with torch.inference_mode():
    output = model(input_tensor)
print(output)
print(output.squeeze().shape)

tensor([[[ 9.4375,  6.8438,  5.0000,  ...,  1.9297,  1.9297,  1.9297],
         [13.0625, 12.3750, 10.0625,  ...,  3.4062,  3.4062,  3.4062],
         [11.2500, 12.0000, 10.6875,  ...,  1.4453,  1.4453,  1.4453],
         [14.6250, 13.3125, 11.9375,  ...,  1.3359,  1.3359,  1.3359]]],
       dtype=torch.bfloat16)
torch.Size([4, 151936])


In [34]:
last_token = output.squeeze()[-1]
print(last_token)
print(torch.argmax(last_token))
print(tokenizer.decode([torch.argmax(last_token).item()]))

tensor([14.6250, 13.3125, 11.9375,  ...,  1.3359,  1.3359,  1.3359],
       dtype=torch.bfloat16)
tensor(304)
 in


In [36]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id = None 
):

    model.eval()
    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)
        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        yield next_token
        token_ids = torch.cat([token_ids, next_token], dim=1)


In [41]:
prompt = "Explain quantum computing"
input_token_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
max_new_tokens = 50
for token in generate_text_basic_stream(model, input_token_ids, max_new_tokens):

    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)

 and its potential applications in cryptography, data storage, and artificial intelligence. Quantum computing is a revolutionary field of computer science that leverages the principles of quantum mechanics to perform computations. Unlike classical computers, which use bits as the smallest unit of information (which

In [42]:
def benchmark_generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
    warmup=True,
):
    import time

    original_device = next(model.parameters()).device
    devices = [torch.device("cpu")]
    if torch.backends.mps.is_available():
        devices.append(torch.device("mps"))
    else:
        print("Skipping MPS: this PyTorch setup does not have MPS available.")

    results = {}

    def synchronize(target_device):
        if target_device.type == "mps":
            torch.mps.synchronize()

    try:
        for target_device in devices:
            model.to(target_device)
            model.eval()
            device_token_ids = token_ids.to(target_device)

            def run_generation():
                generated_count = 0
                for _ in generate_text_basic_stream(
                    model=model,
                    token_ids=device_token_ids,
                    max_new_tokens=max_new_tokens,
                    eos_token_id=eos_token_id,
                ):
                    generated_count += 1
                synchronize(target_device)
                return generated_count

            if warmup:
                run_generation()

            synchronize(target_device)
            start_time = time.perf_counter()
            generated_count = run_generation()
            elapsed_seconds = time.perf_counter() - start_time
            tokens_per_second = generated_count / elapsed_seconds if elapsed_seconds else 0.0

            results[str(target_device)] = {
                "seconds": elapsed_seconds,
                "tokens": generated_count,
                "tokens_per_second": tokens_per_second,
            }
            print(
                f"{target_device}: {elapsed_seconds:.2f} sec, "
                f"{generated_count} tokens, {tokens_per_second:.2f} tokens/sec"
            )
    finally:
        model.to(original_device)

    if "cpu" in results and "mps" in results:
        speedup = results["cpu"]["seconds"] / results["mps"]["seconds"]
        print(f"MPS speedup vs CPU: {speedup:.2f}x")

    return results

In [43]:
benchmark_results = benchmark_generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids,
    max_new_tokens=10,
    eos_token_id=tokenizer.eos_token_id,
)
benchmark_results

cpu: 0.63 sec, 10 tokens, 15.97 tokens/sec
mps: 0.20 sec, 10 tokens, 49.58 tokens/sec
MPS speedup vs CPU: 3.10x


{'cpu': {'seconds': 0.6260738340206444,
  'tokens': 10,
  'tokens_per_second': 15.972556999835032},
 'mps': {'seconds': 0.20171058317646384,
  'tokens': 10,
  'tokens_per_second': 49.57598080637956}}

In [50]:
# add kv cache

from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_with_kv_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    logits = model(token_ids, cache=cache)[:, -1]
    for step in range(max_new_tokens):
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        yield next_token

        if step + 1 < max_new_tokens:
            logits = model(next_token, cache=cache)[:, -1]


In [51]:
prompt = "Explain quantum computing and its applications"
input_token_ids_tensor = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
generated_ids = []

for token in generate_text_basic_stream_with_kv_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)
    generated_ids.append(token.squeeze(0))


 in cryptography, optimization, and machine learning. Quantum computing is a revolutionary field of computer science that leverages the principles of quantum mechanics to perform computations. Unlike classical computers, which use bits as the smallest unit of information (which can be either 0

In [52]:
import time
from statistics import median


def benchmark_kv_cache_generation(
    model,
    token_ids,
    max_new_tokens,
    device,
    eos_token_id=None,
    repeats=3,
    warmup=True,
):
    target_device = torch.device(device)
    if target_device.type == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS is not available in this PyTorch environment.")

    original_device = next(model.parameters()).device
    device_token_ids = token_ids.to(device=target_device, dtype=torch.long)

    def synchronize():
        if target_device.type == "mps":
            torch.mps.synchronize()

    def run_generation():
        generated_count = 0
        for _ in generate_text_basic_stream_with_kv_cache(
            model=model,
            token_ids=device_token_ids,
            max_new_tokens=max_new_tokens,
            eos_token_id=eos_token_id,
        ):
            generated_count += 1
        synchronize()
        return generated_count

    try:
        model.to(target_device)
        model.eval()

        if warmup:
            run_generation()

        durations = []
        token_counts = []
        for _ in range(repeats):
            synchronize()
            start_time = time.perf_counter()
            generated_count = run_generation()
            durations.append(time.perf_counter() - start_time)
            token_counts.append(generated_count)
    finally:
        model.to(original_device)

    median_seconds = median(durations)
    median_tokens = median(token_counts)
    tokens_per_second = median_tokens / median_seconds if median_seconds else 0.0
    result = {
        "device": str(target_device),
        "median_seconds": median_seconds,
        "median_tokens": median_tokens,
        "tokens_per_second": tokens_per_second,
        "trial_seconds": durations,
    }
    print(
        f"device={target_device}: median {median_seconds:.2f} sec, "
        f"{median_tokens} tokens, {tokens_per_second:.2f} tokens/sec"
    )
    return result


if torch.backends.mps.is_available():
    mps_results = benchmark_kv_cache_generation(
        model=model,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        device="mps",
        eos_token_id=tokenizer.eos_token_id,
    )
else:
    print("Skipping device='mps': MPS is not available.")

cpu_results = benchmark_kv_cache_generation(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    device="cpu",
    eos_token_id=tokenizer.eos_token_id,
)


device=mps: median 1.02 sec, 50 tokens, 48.98 tokens/sec
device=cpu: median 1.81 sec, 50 tokens, 27.67 tokens/sec
